# Measuring the risk of a self-financing hedge

## Simulating paths at a public market scale

The generated price process is

$$
S_{k+1}=S_k\exp\!\left[\left(\mu-\frac{\sigma^2}{2}\right)\Delta t+\sigma\sqrt{\Delta t}\,Z_{k+1}\right],
\qquad Z_{k+1}\sim N(0,1).
$$

Here, $S_k$ is the price at date $k$, $\mu$ is the declared drift, $\sigma$ is the annual volatility, and $\Delta t=T/N$ is the time between trades. The variables $Z_{k+1}$ are independent standard normal draws. The volatility scale comes from fixed public NASDAQ Composite observations, while the option paths are generated teaching data.

### Checking moments under a fixed design

In [1]:
from pathlib import Path
import sys
import time
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import torch
from IPython.display import HTML, display as _display

def display(value):
    module = value.__class__.__module__
    if module.startswith('pandas') and hasattr(value, '_repr_html_'):
        html = value._repr_html_()
        return _display(HTML('<div style="display:flex;justify-content:center;">' + html + '</div>'))
    return _display(value)
BOOK_ROOT = Path.cwd().parents[1]
if str(BOOK_ROOT) not in sys.path:
    sys.path.insert(0, str(BOOK_ROOT))
from book_support import ensure_dataset
FINANCIAL_ROOT = BOOK_ROOT
import RiskLabAI
from RiskLabAI.backtest.deep_hedging import DeepHedge, black_scholes_call_delta, entropic_risk, gbm_hedging_paths, hedging_pnl, train_deep_hedge
assert RiskLabAI.__version__ == '3.2.0'
torch.set_num_threads(1)
torch.set_default_dtype(torch.float64)
FIGURES = BOOK_ROOT / 'outputs' / 'figures' / '31_deep_hedging'
TABLES = BOOK_ROOT / 'outputs' / 'tables' / '31_deep_hedging'
FIGURE_DATA = BOOK_ROOT / 'outputs' / 'figure_data' / '31_deep_hedging'
ARTIFACTS = BOOK_ROOT / 'outputs' / 'artifacts' / '31_deep_hedging'
DATA_FILE = ensure_dataset(BOOK_ROOT, 'fred_nasdaq')
for directory in [FIGURES, TABLES, FIGURE_DATA, ARTIFACTS]:
    directory.mkdir(parents=True, exist_ok=True)
PALETTE = {'background': '#111827', 'axes': '#111827', 'text': '#E5E7EB', 'cyan': '#38BDF8', 'amber': '#FBBF24', 'green': '#34D399', 'rose': '#FB7185', 'violet': '#C084FC', 'grid': '#374151'}
plt.rcParams.update({'font.family': 'DejaVu Serif', 'font.size': 11.5, 'figure.facecolor': PALETTE['background'], 'axes.facecolor': PALETTE['axes'], 'axes.edgecolor': PALETTE['text'], 'axes.labelcolor': PALETTE['text'], 'axes.titlecolor': PALETTE['text'], 'xtick.color': PALETTE['text'], 'ytick.color': PALETTE['text'], 'text.color': PALETTE['text'], 'grid.color': PALETTE['grid'], 'grid.linestyle': ':', 'legend.facecolor': PALETTE['axes'], 'legend.edgecolor': PALETTE['grid'], 'savefig.facecolor': PALETTE['background'], 'savefig.transparent': False, 'savefig.dpi': 300})

def rendered_value(value):
    if pd.isna(value):
        return ''
    if isinstance(value, (bool, np.bool_)):
        return 'Pass' if bool(value) else 'Fail'
    if isinstance(value, (int, np.integer)):
        return f'{int(value):,}'
    if isinstance(value, (float, np.floating)):
        value = float(value)
        if value == 0:
            return '0'
        if value.is_integer():
            return f'{int(value):,}'
        if abs(value) < 0.0001 or abs(value) >= 1000000:
            return f'{value:.6g}'
        return f'{value:.6f}'.rstrip('0').rstrip('.')
    return str(value)

def rendered_frame(frame):
    return frame.apply(lambda column: column.map(rendered_value))

def save_table(frame, stem, caption, label, fit=False):
    shown = rendered_frame(frame)
    shown.to_csv(TABLES / f'{stem}.csv', index=False)
    latex = shown.to_latex(index=False, escape=True, caption=caption, label=label, position='htbp')
    if fit:
        latex = latex.replace('\\begin{tabular}', '\\begin{adjustbox}{max width=\\textwidth}\\begin{tabular}', 1).replace('\\end{tabular}', '\\end{tabular}\\end{adjustbox}', 1)
    (TABLES / f'{stem}.tex').write_text(latex, encoding='utf-8')
    display(shown)
    return shown

def finish_axis(axis, grid=True):
    axis.spines['top'].set_visible(False)
    axis.spines['right'].set_visible(False)
    if grid:
        axis.grid(True, alpha=0.65)
    axis.set_axisbelow(True)

def save_figure(figure, stem):
    figure.savefig(FIGURES / f'{stem}.png', dpi=300, bbox_inches='tight', facecolor=PALETTE['background'], transparent=False)
    figure.savefig(FIGURES / f'{stem}.pdf', bbox_inches='tight', facecolor=PALETTE['background'], transparent=False)
    display(figure)
    plt.close(figure)

def public_volatility():
    frame = pd.read_csv(DATA_FILE, parse_dates=['observation_date'])
    frame['NASDAQCOM'] = pd.to_numeric(frame['NASDAQCOM'], errors='coerce')
    design = frame.loc[(frame['observation_date'] >= '1994-01-04') & (frame['observation_date'] <= '2014-12-31')].copy()
    observed = design.dropna(subset=['NASDAQCOM']).copy()
    observed['Log return'] = np.log(observed['NASDAQCOM'] / observed['NASDAQCOM'].shift())
    returns = observed['Log return'].dropna()
    return (frame, design, observed, returns, float(returns.std(ddof=1) * np.sqrt(252)))

def cashflow_components(prices, positions, strike, cost_rate, premium=0.0):
    zero = torch.zeros_like(positions[:, :1])
    trades = torch.diff(torch.cat([zero, positions, zero], dim=1), dim=1)
    gains = (positions * torch.diff(prices, dim=1)).sum(dim=1)
    turnover = (prices * trades.abs()).sum(dim=1)
    costs = cost_rate * turnover
    payoff = torch.relu(prices[:, -1] - strike)
    pnl = premium + gains - costs - payoff
    return {'trades': trades, 'gains': gains, 'turnover': turnover, 'costs': costs, 'payoff': payoff, 'pnl': pnl}

def metric_record(name, prices, positions, strike, cost_rate, risk_aversion):
    components = cashflow_components(prices, positions, strike, cost_rate)
    pnl = components['pnl'].detach()
    cutoff = torch.quantile(pnl, 0.05)
    return {'Strategy': name, 'Entropic risk': entropic_risk(pnl, risk_aversion).item(), 'Mean cash flow': pnl.mean().item(), 'Cash flow standard deviation': pnl.std(unbiased=True).item(), 'Fifth percentile': cutoff.item(), 'Mean loss in worst five percent': -pnl[pnl <= cutoff].mean().item(), 'Mean turnover': components['turnover'].mean().item(), 'Mean trading cost': components['costs'].mean().item(), 'Mean payoff': components['payoff'].mean().item()}
notation = pd.DataFrame([{'Symbol': 'S_k', 'Description': 'Underlying price at date k', 'Unit': 'Normalized price'}, {'Symbol': 'K', 'Description': 'Call strike', 'Unit': 'Normalized price'}, {'Symbol': 'T', 'Description': 'Time to maturity', 'Unit': 'Years'}, {'Symbol': 'N', 'Description': 'Number of trading steps', 'Unit': 'Count'}, {'Symbol': 'phi_k', 'Description': 'Shares held during interval k', 'Unit': 'Shares'}, {'Symbol': 'c', 'Description': 'Proportional trading cost rate', 'Unit': 'Fraction of traded value'}, {'Symbol': 'lambda', 'Description': 'Entropic risk aversion', 'Unit': 'Inverse cash flow unit'}, {'Symbol': 'P', 'Description': 'Terminal hedge cash flow', 'Unit': 'Cash flow unit'}])
save_table(notation, 'deep_hedging_notation', 'Notation for the one asset hedging study.', 'tab:chapter33-notation', fit=True)
assumptions = pd.DataFrame([{'Feature': 'Underlying assets', 'Choice': 'One'}, {'Feature': 'Liability', 'Choice': 'Short European call'}, {'Feature': 'Path law', 'Choice': 'Generated geometric Brownian motion'}, {'Feature': 'Interest and dividend rates', 'Choice': 'Zero'}, {'Feature': 'Trading cost', 'Choice': 'Proportional to traded value'}, {'Feature': 'Final position', 'Choice': 'Fully liquidated'}, {'Feature': 'Risk measure', 'Choice': 'Entropic risk'}, {'Feature': 'Final evaluation', 'Choice': 'Independent generated paths'}])
save_table(assumptions, 'deep_hedging_assumptions', 'Assumptions of the bounded hedging study.', 'tab:chapter33-assumptions', fit=True)
full_public, public_design, public_observed, public_returns, sigma = public_volatility()
data_quality = pd.DataFrame([{'Check': 'Rows in fixed file', 'Value': len(full_public), 'Pass': len(full_public) > 8000}, {'Check': 'Rows in design dates', 'Value': len(public_design), 'Pass': len(public_design) == 5477}, {'Check': 'Observed index levels', 'Value': len(public_observed), 'Pass': len(public_observed) == 5287}, {'Check': 'Observed log returns', 'Value': len(public_returns), 'Pass': len(public_returns) == 5286}, {'Check': 'Duplicate dates', 'Value': public_design['observation_date'].duplicated().sum(), 'Pass': not public_design['observation_date'].duplicated().any()}, {'Check': 'Nonpositive observed levels', 'Value': int((public_observed['NASDAQCOM'] <= 0).sum()), 'Pass': bool((public_observed['NASDAQCOM'] > 0).all())}])
save_table(data_quality, 'deep_hedging_public_data_quality', 'Quality checks for the fixed public design sample.', 'tab:chapter33-public-quality', fit=True)
provenance = pd.DataFrame([{'Item': 'Source', 'Value': 'Federal Reserve Economic Data'}, {'Item': 'Series', 'Value': 'NASDAQCOM'}, {'Item': 'Design start', 'Value': 'January 4, 1994'}, {'Item': 'Design end', 'Value': 'December 31, 2014'}, {'Item': 'Observed levels', 'Value': len(public_observed)}, {'Item': 'Download during execution', 'Value': 'None'}])
save_table(provenance, 'deep_hedging_public_data_provenance', 'Provenance of the public volatility scale.', 'tab:chapter33-public-provenance', fit=True)
return_summary = pd.DataFrame([{'Statistic': 'Observations', 'Value': len(public_returns)}, {'Statistic': 'Mean daily log return', 'Value': public_returns.mean()}, {'Statistic': 'Daily standard deviation', 'Value': public_returns.std(ddof=1)}, {'Statistic': 'Annualized standard deviation', 'Value': sigma}, {'Statistic': 'Minimum', 'Value': public_returns.min()}, {'Statistic': 'First percentile', 'Value': public_returns.quantile(0.01)}, {'Statistic': 'Median', 'Value': public_returns.median()}, {'Statistic': 'Ninety-ninth percentile', 'Value': public_returns.quantile(0.99)}, {'Statistic': 'Maximum', 'Value': public_returns.max()}])
save_table(return_summary, 'deep_hedging_public_return_summary', 'Public daily return description for the fixed design period.', 'tab:chapter33-public-return-summary', fit=True)
design = {'spot': 1.0, 'strike': 1.0, 'maturity_days': 30, 'maturity': 30 / 365, 'volatility': sigma, 'drift': 0.0, 'steps': 20, 'cost_rate': 0.002, 'risk_aversion': 3.0, 'hidden_size': 16, 'learning_rate': 0.003, 'training_paths': 2048, 'evaluation_paths': 8192, 'iterations': 200}
design_table = pd.DataFrame([{'Parameter': key.replace('_', ' ').title(), 'Value': value} for key, value in design.items()])
save_table(design_table, 'deep_hedging_primary_design', 'Declared primary design for the generated hedging study.', 'tab:chapter33-primary-design', fit=True)
generated_contract = pd.DataFrame([{'Item': 'Generated object', 'Value': 'Option price paths and hedge cash flows'}, {'Item': 'Why generation is necessary', 'Value': 'No approved public option panel contains the required synchronized paths and cost rule'}, {'Item': 'Public input', 'Value': 'Annualized volatility scale only'}, {'Item': 'Generated market claim', 'Value': 'None'}, {'Item': 'Primary path seed', 'Value': 33001}, {'Item': 'Download during execution', 'Value': 'None'}])
save_table(generated_contract, 'deep_hedging_generated_data_contract', 'Boundary between the public scale and generated teaching paths.', 'tab:chapter33-generated-contract', fit=True)
prices = gbm_hedging_paths(spot=design['spot'], maturity=design['maturity'], volatility=design['volatility'], drift=design['drift'], steps=design['steps'], paths=design['evaluation_paths'], seed=33001)
twin = gbm_hedging_paths(spot=design['spot'], maturity=design['maturity'], volatility=design['volatility'], drift=design['drift'], steps=design['steps'], paths=design['evaluation_paths'], seed=33001)
terminal_logs = torch.log(prices[:, -1] / design['spot']).numpy()
theoretical_mean = (design['drift'] - design['volatility'] ** 2 / 2) * design['maturity']
theoretical_sd = design['volatility'] * np.sqrt(design['maturity'])
mean_tolerance = 4 * theoretical_sd / np.sqrt(design['evaluation_paths'])
sd_tolerance = 4 * theoretical_sd / np.sqrt(2 * (design['evaluation_paths'] - 1))
moment_checks = pd.DataFrame([{'Quantity': 'Terminal log-return mean', 'Observed': terminal_logs.mean(), 'Target': theoretical_mean, 'Tolerance': mean_tolerance, 'Pass': abs(terminal_logs.mean() - theoretical_mean) <= mean_tolerance}, {'Quantity': 'Terminal log-return standard deviation', 'Observed': terminal_logs.std(ddof=1), 'Target': theoretical_sd, 'Tolerance': sd_tolerance, 'Pass': abs(terminal_logs.std(ddof=1) - theoretical_sd) <= sd_tolerance}, {'Quantity': 'Smallest generated price', 'Observed': prices.min().item(), 'Target': 0, 'Tolerance': 0, 'Pass': bool(torch.all(prices > 0))}])
save_table(moment_checks, 'deep_hedging_gbm_moment_checks', 'Sample checks for the exact generated path law.', 'tab:chapter33-gbm-moments', fit=True)
reproducibility = pd.DataFrame([{'Check': 'Repeated seed gives identical paths', 'Value': bool(torch.equal(prices, twin)), 'Pass': bool(torch.equal(prices, twin))}, {'Check': 'Path array rows', 'Value': prices.shape[0], 'Pass': prices.shape[0] == design['evaluation_paths']}, {'Check': 'Path array columns', 'Value': prices.shape[1], 'Pass': prices.shape[1] == design['steps'] + 1}, {'Check': 'All prices finite', 'Value': bool(torch.isfinite(prices).all()), 'Pass': bool(torch.isfinite(prices).all())}])
save_table(reproducibility, 'deep_hedging_gbm_reproducibility', 'Reproducibility and shape checks for generated paths.', 'tab:chapter33-gbm-reproducibility', fit=True)
public_path_data = public_observed[['observation_date', 'NASDAQCOM']].rename(columns={'observation_date': 'Date', 'NASDAQCOM': 'Index level'})
public_path_data.to_csv(FIGURE_DATA / 'deep_hedging_public_index_path.csv', index=False)
figure, axis = plt.subplots(figsize=(9.0, 5.0), constrained_layout=True)
axis.plot(public_path_data['Date'], public_path_data['Index level'], color=PALETTE['cyan'], linewidth=1.2)
axis.set(title='Public NASDAQ Composite Design Sample', xlabel='Date', ylabel='Index level')
finish_axis(axis)
save_figure(figure, 'deep_hedging_public_index_path')
return_figure_data = pd.DataFrame({'Daily log return': public_returns.to_numpy()})
return_figure_data.to_csv(FIGURE_DATA / 'deep_hedging_public_return_distribution.csv', index=False)
figure, axis = plt.subplots(figsize=(9.0, 5.0), constrained_layout=True)
axis.hist(public_returns, bins=70, color=PALETTE['violet'], alpha=0.9)
axis.set(title='Public Daily Return Distribution', xlabel='Daily log return', ylabel='Days')
finish_axis(axis)
save_figure(figure, 'deep_hedging_public_return_distribution')
days = np.linspace(0, design['maturity_days'], design['steps'] + 1)
fan = pd.DataFrame({'Day': days})
for index in range(40):
    fan[f'Path {index + 1}'] = prices[index].numpy()
fan.to_csv(FIGURE_DATA / 'deep_hedging_generated_paths.csv', index=False)
figure, axis = plt.subplots(figsize=(9.0, 5.2), constrained_layout=True)
for index in range(40):
    axis.plot(days, prices[index].numpy(), color=PALETTE['cyan'], alpha=0.16, linewidth=0.8)
axis.axhline(design['strike'], color=PALETTE['amber'], linewidth=1.4, label='Strike')
axis.set(title='Generated Price Paths over Thirty Days', xlabel='Calendar day', ylabel='Normalized price')
axis.legend()
finish_axis(axis)
save_figure(figure, 'deep_hedging_generated_paths')
terminal_data = pd.DataFrame({'Terminal price': prices[:, -1].numpy()})
terminal_data.to_csv(FIGURE_DATA / 'deep_hedging_terminal_price_distribution.csv', index=False)
figure, axis = plt.subplots(figsize=(9.0, 5.0), constrained_layout=True)
axis.hist(terminal_data['Terminal price'], bins=65, color=PALETTE['green'], alpha=0.9)
axis.axvline(design['strike'], color=PALETTE['amber'], linewidth=1.5, label='Strike')
axis.set(title='Generated Terminal Price Distribution', xlabel='Terminal normalized price', ylabel='Paths')
axis.legend()
finish_axis(axis)
save_figure(figure, 'deep_hedging_terminal_price_distribution')
(ARTIFACTS / 'primary_design.json').write_text(pd.Series(design).to_json(indent=2), encoding='utf-8')


,Symbol,Description,Unit
0,S_k,Underlying price at date k,Normalized price
1,K,Call strike,Normalized price
2,T,Time to maturity,Years
3,N,Number of trading steps,Count
4,phi_k,Shares held during interval k,Shares
5,c,Proportional trading cost rate,Fraction of traded value
6,lambda,Entropic risk aversion,Inverse cash flow unit
7,P,Terminal hedge cash flow,Cash flow unit


,Feature,Choice
0,Underlying assets,One
1,Liability,Short European call
2,Path law,Generated geometric Brownian motion
3,Interest and dividend rates,Zero
4,Trading cost,Proportional to traded value
5,Final position,Fully liquidated
6,Risk measure,Entropic risk
7,Final evaluation,Independent generated paths


,Check,Value,Pass
0,Rows in fixed file,"8,348",Pass
1,Rows in design dates,"5,477",Pass
2,Observed index levels,"5,287",Pass
3,Observed log returns,"5,286",Pass
4,Duplicate dates,0,Pass
5,Nonpositive observed levels,0,Pass


,Item,Value
0,Source,Federal Reserve Economic Data
1,Series,NASDAQCOM
2,Design start,"January 4, 1994"
3,Design end,"December 31, 2014"
4,Observed levels,"5,287"
5,Download during execution,None


,Statistic,Value
0,Observations,"5,286"
1,Mean daily log return,0.000343
2,Daily standard deviation,0.015998
3,Annualized standard deviation,0.253956
4,Minimum,-0.101684
5,First percentile,-0.04438
6,Median,0.001191
7,Ninety-ninth percentile,0.043589
8,Maximum,0.132546


...

<Figure size 900x500 with 1 Axes>

<Figure size 900x500 with 1 Axes>

<Figure size 900x520 with 1 Axes>

<Figure size 900x500 with 1 Axes>

311

## Computing hedge cash flows after trading costs

For positions $\phi_0,\ldots,\phi_{N-1}$, set $\phi_{-1}=0$ before trading and $\phi_N=0$ after final liquidation. The terminal cash flow of a short call is

$$
P=p+\sum_{k=0}^{N-1}\phi_k(S_{k+1}-S_k)
-c\sum_{k=0}^{N}S_k\lvert\phi_k-\phi_{k-1}\rvert
-(S_N-K)^+.
$$

Here, $p$ is the option premium, the first sum is the hedge gain, $c$ is the proportional cost rate, and the final term is the call payoff owed at maturity. The cost sum includes the initial purchase, all later position changes, and final liquidation.

In [2]:
evaluation_prices = prices[:4096]
delta_positions = black_scholes_call_delta(evaluation_prices, strike=design["strike"], maturity=design["maturity"], volatility=design["volatility"])
zero_positions = torch.zeros_like(delta_positions)
half_positions = torch.full_like(delta_positions, 0.5)

cashflow_rows = []
recalculation_rows = []
for name, positions in [("Zero hedge", zero_positions), ("Constant half-share hedge", half_positions), ("Black-Scholes delta", delta_positions)]:
    components = cashflow_components(evaluation_prices, positions, design["strike"], design["cost_rate"])
    released = hedging_pnl(evaluation_prices, positions, strike=design["strike"], cost_rate=design["cost_rate"])
    cashflow_rows.append({
        "Strategy": name,
        "Mean hedge gain": components["gains"].mean().item(),
        "Mean trading cost": components["costs"].mean().item(),
        "Mean payoff": components["payoff"].mean().item(),
        "Mean terminal cash flow": components["pnl"].mean().item(),
    })
    recalculation_rows.append({
        "Strategy": name,
        "Maximum absolute difference": torch.max(torch.abs(released - components["pnl"])).item(),
        "Pass": bool(torch.allclose(released, components["pnl"], atol=1e-13, rtol=0)),
    })
cashflow_summary = pd.DataFrame(cashflow_rows)
save_table(cashflow_summary, "deep_hedging_cashflow_summary", "Cash flow components under three fixed hedges.", "tab:chapter33-cashflow-summary", fit=True)
save_table(pd.DataFrame(recalculation_rows), "deep_hedging_cashflow_recalculation", "Independent cash flow recomputation against the released function.", "tab:chapter33-cashflow-recalculation", fit=True)

delta_components = cashflow_components(evaluation_prices, delta_positions, design["strike"], design["cost_rate"])
zero = torch.zeros_like(delta_positions[:, :1])
delta_trades = torch.diff(torch.cat([zero, delta_positions, zero], dim=1), dim=1)
cost_breakdown = pd.DataFrame([
    {"Cost part": "Initial purchase", "Mean cost": (design["cost_rate"] * evaluation_prices[:, 0] * delta_trades[:, 0].abs()).mean().item()},
    {"Cost part": "Later changes", "Mean cost": (design["cost_rate"] * evaluation_prices[:, 1:-1] * delta_trades[:, 1:-1].abs()).sum(dim=1).mean().item()},
    {"Cost part": "Terminal liquidation", "Mean cost": (design["cost_rate"] * evaluation_prices[:, -1] * delta_trades[:, -1].abs()).mean().item()},
    {"Cost part": "Total", "Mean cost": delta_components["costs"].mean().item()},
])
save_table(cost_breakdown, "deep_hedging_cost_breakdown", "Mean Black-Scholes delta cost from initial purchase through liquidation.", "tab:chapter33-cost-breakdown", fit=True)

path_index = 0
path_table = pd.DataFrame({
    "Date": np.arange(design["steps"] + 1),
    "Price": evaluation_prices[path_index].numpy(),
    "Holding before next move": np.append(delta_positions[path_index].numpy(), np.nan),
    "Trade": delta_trades[path_index].numpy(),
    "Trading cost": (design["cost_rate"] * evaluation_prices[path_index] * delta_trades[path_index].abs()).numpy(),
})
save_table(path_table, "deep_hedging_single_path_cashflows", "One generated path with delta holdings, trades, and costs.", "tab:chapter33-single-path", fit=True)

payoff_prices = np.linspace(0.6, 1.4, 241)
payoff_data = pd.DataFrame({"Terminal price": payoff_prices, "Call payoff": np.maximum(payoff_prices - design["strike"], 0)})
payoff_data.to_csv(FIGURE_DATA / "deep_hedging_call_payoff.csv", index=False)
figure, axis = plt.subplots(figsize=(9.0, 5.0), constrained_layout=True)
axis.plot(payoff_data["Terminal price"], payoff_data["Call payoff"], color=PALETTE["amber"], linewidth=2.0)
axis.axvline(design["strike"], color=PALETTE["text"], linewidth=1.0, alpha=0.8)
axis.set(title="European Call Payoff", xlabel="Terminal normalized price", ylabel="Payoff")
finish_axis(axis)
save_figure(figure, "deep_hedging_call_payoff")

cashflow_data = pd.DataFrame({
    "Zero hedge": cashflow_components(evaluation_prices, zero_positions, design["strike"], design["cost_rate"])["pnl"].numpy(),
    "Constant half-share hedge": cashflow_components(evaluation_prices, half_positions, design["strike"], design["cost_rate"])["pnl"].numpy(),
    "Black-Scholes delta": delta_components["pnl"].numpy(),
})
cashflow_data.to_csv(FIGURE_DATA / "deep_hedging_fixed_cashflow_distributions.csv", index=False)
figure, axis = plt.subplots(figsize=(9.0, 5.2), constrained_layout=True)
for name, color in [("Zero hedge", PALETTE["rose"]), ("Constant half-share hedge", PALETTE["violet"]), ("Black-Scholes delta", PALETTE["cyan"])]:
    axis.hist(cashflow_data[name], bins=65, density=True, histtype="step", linewidth=1.6, label=name, color=color)
axis.set(title="Terminal Cash Flows under Fixed Hedges", xlabel="Terminal cash flow", ylabel="Density")
axis.legend()
finish_axis(axis)
save_figure(figure, "deep_hedging_fixed_cashflow_distributions")

,Strategy,Mean hedge gain,Mean trading cost,Mean payoff,Mean terminal cash flow
0,Zero hedge,0,0,0.029511,-0.029511
1,Constant half-share hedge,0.000236,0.002,0.029511,-0.031275
2,Black-Scholes delta,0.000438,0.004803,0.029511,-0.033876


,Strategy,Maximum absolute difference,Pass
0,Zero hedge,0,Pass
1,Constant half-share hedge,0,Pass
2,Black-Scholes delta,0,Pass


,Cost part,Mean cost
0,Initial purchase,0.001029
1,Later changes,0.00274
2,Terminal liquidation,0.001034
3,Total,0.004803


,Date,Price,Holding before next move,Trade,Trading cost
0,0,1,0.51452,0.51452,0.001029
1,1,0.98732,0.442614,-0.071905,0.000142
2,2,0.983595,0.418806,-0.023808,4.68349e-05
3,3,0.991931,0.465285,0.046479,9.22078e-05
4,4,0.992362,0.466058,0.000773,1.53375e-06
5,5,1.006954,0.556239,0.090181,0.000182
6,6,1.000537,0.51566,-0.040578,8.12004e-05
7,7,0.995218,0.479138,-0.036522,7.26954e-05
8,8,0.98644,0.415313,-0.063825,0.000126
9,9,0.971778,0.307415,-0.107898,0.00021


<Figure size 900x500 with 1 Axes>

<Figure size 900x520 with 1 Axes>

### Entropic Risk

The chapter evaluates a terminal cash flow $P$ with

$$
\rho_{\lambda}(P)=\frac{1}{\lambda}\log E\!\left[e^{-\lambda P}\right],
\qquad \lambda>0.
$$

The parameter $\lambda$ controls how strongly low cash flows influence the result. A larger value gives more weight to adverse outcomes. This is one declared risk preference, not a universal measure of investment quality.

In [3]:
risk_rows = []
delta_pnl = delta_components["pnl"]
for value in [0.1, 0.5, 1.0, 3.0, 10.0]:
    anchor = delta_pnl.min().item()
    independent = -anchor + np.log(np.exp(-value * (delta_pnl.numpy() - anchor)).mean()) / value
    released = entropic_risk(delta_pnl, value).item()
    risk_rows.append({"Risk aversion": value, "Released value": released, "Independent value": independent, "Absolute difference": abs(released - independent), "Translation residual": abs(entropic_risk(delta_pnl + 50000, value).item() - (released - 50000))})
entropic_checks = pd.DataFrame(risk_rows)
save_table(entropic_checks, "deep_hedging_entropic_risk_checks", "Stable and independently recomputed entropic-risk values.", "tab:chapter33-entropic-checks", fit=True)

invalid_cases = [
    ("Zero spot", lambda: gbm_hedging_paths(spot=0, maturity=1, volatility=0.2, drift=0, steps=2, paths=2, seed=1)),
    ("Zero maturity", lambda: gbm_hedging_paths(spot=1, maturity=0, volatility=0.2, drift=0, steps=2, paths=2, seed=1)),
    ("Negative volatility", lambda: gbm_hedging_paths(spot=1, maturity=1, volatility=-0.2, drift=0, steps=2, paths=2, seed=1)),
    ("Zero steps", lambda: gbm_hedging_paths(spot=1, maturity=1, volatility=0.2, drift=0, steps=0, paths=2, seed=1)),
    ("Zero path count", lambda: gbm_hedging_paths(spot=1, maturity=1, volatility=0.2, drift=0, steps=2, paths=0, seed=1)),
    ("Zero risk aversion", lambda: entropic_risk(torch.ones(2), 0)),
    ("Zero delta volatility", lambda: black_scholes_call_delta(torch.ones((2, 3)), strike=1, maturity=1, volatility=0)),
]
invalid_rows = []
for name, function in invalid_cases:
    try:
        function()
        invalid_rows.append({"Case": name, "Rejected": False, "Result": "Accepted"})
    except Exception as error:
        invalid_rows.append({"Case": name, "Rejected": isinstance(error, ValueError), "Result": "Invalid input"})
invalid_checks = pd.DataFrame(invalid_rows)
assert invalid_checks["Rejected"].all()
save_table(invalid_checks, "deep_hedging_invalid_input_checks", "Invalid inputs rejected by the released calculations.", "tab:chapter33-invalid-inputs", fit=True)

risk_curve = entropic_checks[["Risk aversion", "Released value"]].rename(columns={"Released value": "Entropic risk"})
risk_curve.to_csv(FIGURE_DATA / "deep_hedging_entropic_risk_curve.csv", index=False)
figure, axis = plt.subplots(figsize=(9.0, 5.0), constrained_layout=True)
axis.plot(risk_curve["Risk aversion"], risk_curve["Entropic risk"], color=PALETTE["rose"], marker="o", linewidth=1.8)
axis.set_xscale("log")
axis.set(title="Entropic Risk across Declared Risk Aversion", xlabel="Risk aversion", ylabel="Entropic risk")
finish_axis(axis)
save_figure(figure, "deep_hedging_entropic_risk_curve")

,Risk aversion,Released value,Independent value,Absolute difference,Translation residual
0,0.1,0.033877,0.033877,1.52101e-14,0
1,0.5,0.033884,0.033884,3.11556e-15,0
2,1,0.033893,0.033893,6.52256e-16,0
3,3,0.033928,0.033928,4.57967e-16,0
4,10,0.034052,0.034052,6.245e-17,0


,Case,Rejected,Result
0,Zero spot,Pass,Invalid input
1,Zero maturity,Pass,Invalid input
2,Negative volatility,Pass,Invalid input
3,Zero steps,Pass,Invalid input
4,Zero path count,Pass,Invalid input
5,Zero risk aversion,Pass,Invalid input
6,Zero delta volatility,Pass,Invalid input


<Figure size 900x500 with 1 Axes>